In [4]:
# notebooks/04_model_comparison.ipynb

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from xgboost import XGBClassifier

# --------------------------
# 1. Load & preprocess data
# --------------------------
df = pd.read_csv("../data/Telco-Customer-Churn.csv")
df = df.drop("customerID", axis=1)

# Fix TotalCharges (convert to numeric, drop blanks)
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
df = df.dropna()

# Encode Yes/No binary columns
binary_map = {"Yes": 1, "No": 0}
for col in df.columns:
    if set(df[col].unique()) <= {"Yes", "No"}:
        df[col] = df[col].map(binary_map)

# One-hot encode remaining categoricals
categorical_cols = df.select_dtypes(include="object").columns
df = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

# Scale numeric features
scaler = StandardScaler()
numeric_cols = ["tenure", "MonthlyCharges", "TotalCharges"]
df[numeric_cols] = scaler.fit_transform(df[numeric_cols])

# Train/test split
X = df.drop("Churn", axis=1)
y = df["Churn"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# --------------------------
# 2. Random Forest
# --------------------------
rf = RandomForestClassifier(n_estimators=200, random_state=42)
rf.fit(X_train, y_train)

rf_pred = rf.predict(X_test)
rf_prob = rf.predict_proba(X_test)[:, 1]

print("\n--- Random Forest ---")
print("Accuracy:", accuracy_score(y_test, rf_pred))
print("Precision:", precision_score(y_test, rf_pred))
print("Recall:", recall_score(y_test, rf_pred))
print("F1-score:", f1_score(y_test, rf_pred))
print("ROC-AUC:", roc_auc_score(y_test, rf_prob))

# --------------------------
# 3. XGBoost
# --------------------------
xgb = XGBClassifier(
    n_estimators=300,
    learning_rate=0.1,
    max_depth=5,
    subsample=0.8,
    colsample_bytree=0.8,
    use_label_encoder=False,
    eval_metric="logloss",
    random_state=42
)
xgb.fit(X_train, y_train)

xgb_pred = xgb.predict(X_test)
xgb_prob = xgb.predict_proba(X_test)[:, 1]

print("\n--- XGBoost ---")
print("Accuracy:", accuracy_score(y_test, xgb_pred))
print("Precision:", precision_score(y_test, xgb_pred))
print("Recall:", recall_score(y_test, xgb_pred))
print("F1-score:", f1_score(y_test, xgb_pred))
print("ROC-AUC:", roc_auc_score(y_test, xgb_prob))



--- Random Forest ---
Accuracy: 0.7938877043354655
Precision: 0.6418918918918919
Recall: 0.5080213903743316
F1-score: 0.5671641791044776
ROC-AUC: 0.8193867091851261


/opt/conda/lib/python3.12/site-packages/xgboost/training.py:183: UserWarning: [18:20:05] WARNING: /croot/xgboost-split_1749630910898/work/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)



--- XGBoost ---
Accuracy: 0.7711442786069652
Precision: 0.5764705882352941
Recall: 0.5240641711229946
F1-score: 0.5490196078431373
ROC-AUC: 0.8159208680391985
